In [ ]:
import os

INPUT_DIR = '/kaggle/input/datasets/shreyash1110/acne04-yolov8' # Adjust if the folder name is slightly different

def print_directory_tree(startpath, max_depth=2):
    print(f"Scanning: {startpath}")
    for root, dirs, files in os.walk(startpath):
        depth = root[len(startpath):].count(os.sep)
        if depth > max_depth: continue
        indent = ' ' * 4 * depth
        print(f"{indent}📁 {os.path.basename(root)}/ ({len(files)} files)")
        if files:
            for f in files[:3]: print(f"{indent}    📄 {f}")
            if len(files) > 3: print(f"{indent}    ... and {len(files) - 3} more.")

print_directory_tree(INPUT_DIR)

In [ ]:
import os

# Define our new paths based on your screenshot
BASE_DIR = '/kaggle/input/datasets/shreyash1110/acne04-yolov8/dataset'
TRAIN_IMG_DIR = os.path.join(BASE_DIR, 'Images/train')
TRAIN_LBL_DIR = os.path.join(BASE_DIR, 'labels/train')

# Get the first text file
sample_label_file = os.listdir(TRAIN_LBL_DIR)[0]
sample_label_path = os.path.join(TRAIN_LBL_DIR, sample_label_file)

print(f"Reading label file: {sample_label_file}\n")
with open(sample_label_path, 'r') as f:
    lines = f.readlines()
    for i, line in enumerate(lines):
        print(line.strip())
        if i >= 4: # Print only the first 5 lesions to avoid spamming the output
            print("... and more lesions")
            break
            
print(f"\nTotal lesions in this image: {len(lines)}")

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

def plot_yolo_bounding_boxes(image_path, label_path):
    # 1. Load the image using OpenCV (BGR format) and convert to RGB
    img = cv2.imread(image_path)
    if img is None:
        print(f"Error loading image: {image_path}")
        return
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_height, img_width = img.shape[:2]

    # 2. Read the labels
    try:
        with open(label_path, 'r') as f:
            lines = f.readlines()
    except FileNotFoundError:
        print(f"No label file found for {image_path}")
        return

    # 3. Draw each bounding box
    for line in lines:
        parts = line.strip().split()
        if len(parts) != 5: continue
            
        class_id = int(parts[0])
        x_center_norm, y_center_norm, w_norm, h_norm = map(float, parts[1:])

        # Convert YOLO normalized coordinates back to absolute pixel values
        x_center = x_center_norm * img_width
        y_center = y_center_norm * img_height
        box_width = w_norm * img_width
        box_height = h_norm * img_height

        # Calculate top-left and bottom-right coordinates for OpenCV
        x_min = int(x_center - (box_width / 2))
        y_min = int(y_center - (box_height / 2))
        x_max = int(x_center + (box_width / 2))
        y_max = int(y_center + (box_height / 2))

        # Draw the rectangle (Red color, 2px thickness)
        cv2.rectangle(img, (x_min, y_min), (x_max, y_max), (255, 0, 0), 2)
        
        # Put the class label text
        cv2.putText(img, f"Class {class_id}", (x_min, y_min - 5), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

    # 4. Plot the image
    plt.figure(figsize=(12, 12))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Ground Truth Bounding Boxes\n{os.path.basename(image_path)} ({len(lines)} lesions)")
    plt.show()

# Let's visualize the exact image corresponding to the label file we looked at in Step 4
sample_img_file = sample_label_file.replace('.txt', '.jpg')
sample_img_path = os.path.join(TRAIN_IMG_DIR, sample_img_file)

plot_yolo_bounding_boxes(sample_img_path, sample_label_path)

# Data Engineering Pipeline

<h3>Step 1: The Box Tightening Pipeline<h3/>

In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.1 MB/s eta 0:00:00a 0:00:01


In [2]:
import os
import shutil

# Paths based on your dataset structure
ORIGINAL_IMG_DIR = '/kaggle/input/datasets/shreyash1110/acne04-yolov8/dataset/Images/train'
ORIGINAL_LBL_DIR = '/kaggle/input/datasets/shreyash1110/acne04-yolov8/dataset/labels/train'

# Our new clean working directories
WORKING_DIR = '/kaggle/working/skinwise_data'
NEW_IMG_DIR = os.path.join(WORKING_DIR, 'images/train')
NEW_LBL_DIR = os.path.join(WORKING_DIR, 'labels/train')

# Create directories
os.makedirs(NEW_IMG_DIR, exist_ok=True)
os.makedirs(NEW_LBL_DIR, exist_ok=True)

SHRINK_FACTOR = 0.85 # Shrink boxes by 15%

def tighten_yolo_boxes():
    label_files = [f for f in os.listdir(ORIGINAL_LBL_DIR) if f.endswith('.txt')]
    print(f"Processing {len(label_files)} label files...")
    
    processed_count = 0
    for label_file in label_files:
        orig_label_path = os.path.join(ORIGINAL_LBL_DIR, label_file)
        new_label_path = os.path.join(NEW_LBL_DIR, label_file)
        
        # We also need to copy the corresponding image to our new directory
        img_file = label_file.replace('.txt', '.jpg')
        orig_img_path = os.path.join(ORIGINAL_IMG_DIR, img_file)
        new_img_path = os.path.join(NEW_IMG_DIR, img_file)
        
        if not os.path.exists(orig_img_path):
            continue # Skip if image is missing
            
        # Copy image
        shutil.copy(orig_img_path, new_img_path)
        
        # Read, shrink, and write labels
        new_lines = []
        with open(orig_label_path, 'r') as f:
            lines = f.readlines()
            for line in lines:
                parts = line.strip().split()
                if len(parts) == 5:
                    class_id = parts[0]
                    cx, cy, w, h = map(float, parts[1:])
                    
                    # Apply shrinkage to width and height only
                    new_w = w * SHRINK_FACTOR
                    new_h = h * SHRINK_FACTOR
                    
                    new_lines.append(f"{class_id} {cx:.6f} {cy:.6f} {new_w:.6f} {new_h:.6f}\n")
                    
        with open(new_label_path, 'w') as f:
            f.writelines(new_lines)
            
        processed_count += 1
        
    print(f"✅ Successfully engineered {processed_count} images and labels with 15% tighter boxes!")

tighten_yolo_boxes()


Processing 1165 label files...
✅ Successfully engineered 1165 images and labels with 15% tighter boxes!


<h3>Step 2: Creating the YAML Configuration</h3>

In [3]:
yaml_content = f"""
path: /kaggle/working/skinwise_data
train: images/train
val: images/train # We are using train for val temporarily just to test the pipeline

names:
  0: acne
"""

with open('/kaggle/working/skinwise_data.yaml', 'w') as f:
    f.write(yaml_content)

print("✅ data.yaml created successfully!")


✅ data.yaml created successfully!


<h3>Step 3: Train the Baseline YOLOv8 Model!<h3/>

In [ ]:
# Install ultralytics (YOLOv8)
!pip install ultralytics

from ultralytics import YOLO

# Load a pre-trained YOLOv8 'small' model
model = YOLO('yolov8s.pt') 

print("🚀 Starting YOLOv8 Training...")
# Train the model on our newly engineered data
results = model.train(
    data='/kaggle/working/skinwise_data.yaml', 
    epochs=20,          # Just 20 epochs for our baseline test
    imgsz=640,          # Resize images to 640x640
    batch=16,           # Batch size of 16 fits nicely on Kaggle GPUs
    project='/kaggle/working/SkinWISE_Models',
    name='baseline_yolov8s'
)
print("✅ Baseline Training Complete!")

<h2>Phase 2: Model Export & Sync
</h2>

<h3>Step 1: Exporting to ONNX</h3>

In [ ]:
from ultralytics import YOLO

# Load the best weights from our baseline training run
model = YOLO('/kaggle/working/SkinWISE_Models/baseline_yolov8s/weights/best.pt')

print("⚙️ Exporting model to ONNX format...")
# Export the model. imgsz must match what we will receive from the web frontend
success = model.export(
    format='onnx', 
    dynamic=False, # Fixed size is faster for CPU
    imgsz=640,
    opset=17       # Industry standard ONNX opset
)

print(f"✅ Export complete: {success}")

<h3>Step 2: Packaging for Local Download</h3>

In [ ]:
import shutil

# Zip the training results and the ONNX model
shutil.make_archive(
    '/kaggle/working/baseline_results', 
    'zip', 
    '/kaggle/working/SkinWISE_Models/baseline_yolov8s'
)
print("📦 baseline_results.zip created in your /kaggle/working/ directory.")
print("⬇️ You can now download it from the right-hand panel in Kaggle.")


# Phase 3: Convergence


the baseline model failed because it was only trained for 20 epochs without augmentation. We are now going to train it for 150 epochs and turn on heavy data augmentation (hue shifts, rotation, mosaic blending). This forces the neural network to learn what a pimple looks like under terrible lighting and weird angles, drastically boosting its confidence scores.

In [ ]:
!pip install ultralytics

In [ ]:
import os
print(os.path.exists('/kaggle/working/skinwise_data.yaml'))

In [ ]:
from ultralytics import YOLO
import os

# Hard check to prevent training if data engineering failed
YAML_PATH = '/kaggle/working/skinwise_data.yaml'
if not os.path.exists(YAML_PATH):
    raise FileNotFoundError(f"❌ Stop! {YAML_PATH} is missing. You must run Step 1 and 2 first!")

print("🚀 Starting Phase 3: Production Training (150 Epochs)")

# Load the raw pretrained weights
model = YOLO('yolov8s.pt') 

# Train with heavy augmentation
results = model.train(
    data=YAML_PATH,           # GUARANTEED CORRECT PATH
    epochs=150,               
    imgsz=640,
    batch=16,
    patience=30,              
    project='/kaggle/working/SkinWISE_Models',
    name='production_yolov8s',
    
    # Heavy Data Augmentation 
    hsv_h=0.015,              
    hsv_s=0.7,                
    hsv_v=0.4,                
    degrees=15.0,             
    fliplr=0.5,               
    mosaic=1.0,               
    mixup=0.15                
)

print("✅ Production Training Complete!")

In [ ]:
from ultralytics import YOLO
import shutil

# 1. Load the newly trained production weights
model = YOLO('/kaggle/working/SkinWISE_Models/production_yolov8s/weights/best.pt')

print("⚙️ Exporting production model to ONNX format...")
# 2. Export the model
success = model.export(
    format='onnx', 
    dynamic=False, 
    imgsz=640,
    opset=17       
)
print(f"✅ Export complete: {success}")

# 3. Zip the results for easy download
shutil.make_archive(
    '/kaggle/working/production_results', 
    'zip', 
    '/kaggle/working/SkinWISE_Models/production_yolov8s'
)
print("📦 production_results.zip created in your /kaggle/working/ directory.")
print("⬇️ You can now download it from the right-hand panel in Kaggle.")

<h2>High-Resolution (Small Object) Training Pipeline</h2>

In [4]:
from ultralytics import YOLO
import os

YAML_PATH = '/kaggle/working/skinwise_data.yaml'
if not os.path.exists(YAML_PATH):
    raise FileNotFoundError("❌ Pipeline Failure: YAML not found. Run Step 1 data engineering first.")

print("🚀 Starting High-Resolution SOD Training (imgsz=1024)")

# Initialize fresh architecture weights
model = YOLO('yolov8s.pt') 

results = model.train(
    data=YAML_PATH,
    epochs=150,               
    imgsz=1024,               # ⬆️ Increased spatial resolution for micro-lesions
    batch=8,                  # ⬇️ Decreased to prevent T4 CUDA OOM
    patience=30,              
    project='/kaggle/working/SkinWISE_Models',
    name='production_yolov8s_highres',
    
    # Standard Augmentations
    hsv_h=0.015,              
    hsv_s=0.7,                
    hsv_v=0.4,                
    degrees=15.0,             
    fliplr=0.5,               
    mixup=0.15,
    
    # Small Object Detection (SOD) Specific Tweaks
    mosaic=1.0,               
    close_mosaic=20           # 🛑 Disable mosaic for the last 20 epochs to learn true scale
)

print("✅ High-Resolution Training Complete!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🚀 Starting High-Resolution SOD Training (imgsz=1024)
Ultralytics 8.4.52 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=20, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/skinwise_data.yaml, degrees=15.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, fr

In [5]:
from ultralytics import YOLO

print("📊 Running explicit validation on the held-out set...")

# Load the newly trained best weights
model = YOLO('/kaggle/working/SkinWISE_Models/production_yolov8s_highres/weights/best.pt')

# Run validation. This will output the mAP50 and per-class metrics to the console.
metrics = model.val()

print("\n✅ Validation complete. Check the console output above for the final mAP50.")

📊 Running explicit validation on the held-out set...
Ultralytics 8.4.52 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4589.0±872.4 MB/s, size: 791.5 KB)
val: Scanning /kaggle/working/skinwise_data/labels/train.cache... 1165 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1165/1165 407.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 73/73 1.5it/s 50.3s0.5ss
                   all       1165      14792      0.544      0.509      0.472      0.145
Speed: 2.0ms preprocess, 22.4ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to /kaggle/working/runs/detect/val

✅ Validation complete. Check the console output above for the final mAP50.


In [6]:
print("⚙️ Exporting high-resolution model to ONNX...")

# Exporting at 1024x1024 to preserve spatial resolution of micro-lesions
success = model.export(
    format='onnx', 
    dynamic=False, 
    imgsz=1024,    # Must match our training resolution
    opset=17
)

print(f"✅ Export complete. ONNX model saved at: {success}")

⚙️ Exporting high-resolution model to ONNX...
Ultralytics 8.4.52 🚀 Python-3.12.12 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from '/kaggle/working/SkinWISE_Models/production_yolov8s_highres/weights/best.pt' with input shape (1, 3, 1024, 1024) BCHW and output shape(s) (1, 5, 21504) (21.5 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 12 packages in 282ms
Prepared 2 packages in 3.10s
Installed 2 packages in 15ms
 + onnxruntime-gpu==1.26.0
 + onnxslim==0.1.93

requirements: AutoUpdate success ✅ 3.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.20.1 opset 17...
ONNX: slimming with onnxslim 0.1.93...
ONNX: export success ✅ 6.4s, s

In [7]:
import shutil
import os

ARCHIVE_NAME = '/kaggle/working/highres_production_results'
SOURCE_DIR = '/kaggle/working/SkinWISE_Models/production_yolov8s_highres'

print(f"📦 Packaging model artifacts from {SOURCE_DIR}...")

if os.path.exists(SOURCE_DIR):
    shutil.make_archive(
        ARCHIVE_NAME, 
        'zip', 
        SOURCE_DIR
    )
    print(f"✅ Zipped successfully.")
    print(f"⬇️ Download '{ARCHIVE_NAME}.zip' from the Kaggle right-hand output panel.")
else:
    print("❌ Error: Source directory not found. Did the training complete successfully?")

📦 Packaging model artifacts from /kaggle/working/SkinWISE_Models/production_yolov8s_highres...
✅ Zipped successfully.
⬇️ Download '/kaggle/working/highres_production_results.zip' from the Kaggle right-hand output panel.


In [9]:
from ultralytics import YOLO
from pathlib import Path
import numpy as np

# =========================================================
# CONFIG
# =========================================================

MODEL_PATH = Path(
    "/kaggle/working/SkinWISE_Models/production_yolov8s_highres/weights/best.pt"
)

DATA_PATH = Path("/kaggle/working/skinwise_data.yaml")

# =========================================================
# SAFETY CHECKS
# =========================================================

assert MODEL_PATH.exists(), f"❌ Model not found: {MODEL_PATH}"
assert DATA_PATH.exists(), f"❌ Data config not found: {DATA_PATH}"

print("📦 Loading Model...")
model = YOLO(str(MODEL_PATH))

print("📊 Running Validation Evaluation...")

# =========================================================
# VALIDATION
# =========================================================

metrics = model.val(
    data=str(DATA_PATH),
    imgsz=1024,      # Match training resolution
    split="val",
    save_json=True,
    plots=True,
    verbose=True
)

# =========================================================
# METRICS EXTRACTION
# =========================================================

precision = float(metrics.box.mp)
recall = float(metrics.box.mr)
map50 = float(metrics.box.map50)
map50_95 = float(metrics.box.map)

# F1-score calculation
f1_curve = metrics.box.f1
max_f1_index = np.argmax(f1_curve)
max_f1_score = float(f1_curve[max_f1_index])

# False Negative Rate
false_negative_rate = 1 - recall

# =========================================================
# REPORT
# =========================================================

print("\n" + "=" * 60)
print("🎯 FINAL EVALUATION METRICS")
print("=" * 60)

print(f"📌 Model Path:")
print(f"   {MODEL_PATH}")

print("\n📊 Detection Performance")
print(f"   Mean Precision : {precision:.4f}")
print(f"   Mean Recall    : {recall:.4f}")
print(f"   mAP@50         : {map50:.4f}")
print(f"   mAP@50-95      : {map50_95:.4f}")
print(f"   Max F1-Score   : {max_f1_score:.4f}")

print("\n🚨 Error Analysis")
print(f"   False Negative Rate : {false_negative_rate:.4f}")
print(f"   Missed Lesions      : {false_negative_rate * 100:.2f}%")

print("=" * 60)

# =========================================================
# ENGINEERING INTERPRETATION
# =========================================================

if recall < 0.50:
    print("❌ CRITICAL: Recall is too low for production deployment.")
elif recall < 0.70:
    print("⚠️ WARNING: Model still misses a significant number of lesions.")
else:
    print("✅ Recall is approaching deployable quality.")

if false_negative_rate > 0.40:
    print("⚠️ Small Object Detection issue still exists.")

print("\n✅ Evaluation Complete")

📦 Loading Model...
📊 Running Validation Evaluation...
Ultralytics 8.4.52 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3467.8±1805.5 MB/s, size: 667.9 KB)
val: Scanning /kaggle/working/skinwise_data/labels/train.cache... 1165 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1165/1165 305.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 73/73 1.4it/s 52.9s0.5ss
                   all       1165      14792      0.544      0.509      0.472      0.145
Speed: 2.4ms preprocess, 22.3ms inference, 0.0ms loss, 1.5ms postprocess per image
Saving /kaggle/working/runs/detect/val-2/predictions.json...
Results saved to /kaggle/working/runs/detect/val-2

🎯 FINAL EVALUATION METRICS
📌 Model Path:
   /kaggle/working/SkinWISE_Models/production_yolov8s_highres/weights/best.pt

📊 Detect